# Electric forecast historical data scraping

## Import libraries

In [6]:
import os
import time
import requests
import zipfile
from io import StringIO, BytesIO
from bs4 import BeautifulSoup
import polars as pl
import numpy as np

## Scrape historical data

### Tokyo

In [2]:
BASE_URL = "https://www.tepco.co.jp/forecast/html/images"
OUTPUT_PATH = r"/workspace/src/stg/data_lake/electric_forecast/tokyo"

#### Scrape raw csv files

In [4]:
for year in range(2016, 2023):
    file_name = f"juyo-{year}.csv"
    url = f"{BASE_URL}/{file_name}"
    response = requests.get(url)

    if response.status_code == 200:
        print(f"Processing data for year {year}")
        df = pl.read_csv(source=url, encoding="cp932", skip_rows=2)
        df.write_parquet(os.path.join(OUTPUT_PATH, f"juyo-{year}.parquet"))
        print(f"Saved parquet for year {year}")
    else:
        print(f"Failed to retrieve data for year {year}")
        continue

    time.sleep(np.random.uniform(1, 3))

Processing data for year 2016
Saved parquet for year 2016
Processing data for year 2017
Saved parquet for year 2017
Processing data for year 2018
Saved parquet for year 2018
Processing data for year 2019
Saved parquet for year 2019
Processing data for year 2020
Saved parquet for year 2020
Processing data for year 2021
Saved parquet for year 2021
Processing data for year 2022
Saved parquet for year 2022


In [5]:
tokyo_parquet = r"/workspace/src/stg/data_lake/electric_forecast/tokyo/juyo-2022.parquet"
df_tokyo_2022 = pl.read_parquet(tokyo_parquet)
df_tokyo_2022.head()

DATE,TIME,実績(万kW)
str,str,i64
"""2022/1/1""","""0:00""",3266
"""2022/1/1""","""1:00""",3062
"""2022/1/1""","""2:00""",2929
"""2022/1/1""","""3:00""",2828
"""2022/1/1""","""4:00""",2786


#### Scrape Zip files

In [ ]:
ZIP_PATH = r"/workspace/src/stg/data_lake/electric_forecast/tokyo/zip"
year_list = [i for i in range(2022,2026)]
month_list = [i for i in range(1,13)]
print(year_list,month_list)

[2022, 2023, 2024, 2025] [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]


In [ ]:
#for year in year_list:
#for month in range(2,13):
zip_file_name = "202512_power_usage.zip"
zip_url = f"{BASE_URL}/{zip_file_name}"
response = requests.get(zip_url)

if response.status_code == 200:
    with open(os.path.join(ZIP_PATH, zip_file_name), 'wb') as f:
        f.write(response.content)
    print(f"Downloaded {zip_file_name}")
else:
    print(f"Failed to download {zip_file_name}")

    time.sleep(np.random.uniform(3, 6))

Downloaded 202502_power_usage.zip
Downloaded 202503_power_usage.zip
Downloaded 202504_power_usage.zip
Downloaded 202505_power_usage.zip
Downloaded 202506_power_usage.zip
Downloaded 202507_power_usage.zip
Downloaded 202508_power_usage.zip
Downloaded 202509_power_usage.zip
Downloaded 202510_power_usage.zip
Downloaded 202511_power_usage.zip
Downloaded 202512_power_usage.zip


### Hokkaido

In [16]:
HKD_PATH = r"/workspace/src/stg/data_lake/electric_forecast/hokkaido"
HKD_BASE_URL = "https://denkiyoho.hepco.co.jp/area/data/zip"
QUARTER = {
    1:"04-06",
    2:"07-09",
    3:"10-12",
    4:"01-03"
}
HKD_YEAR_LIST = [i for i in range(2020,2026)]

In [17]:
for year in HKD_YEAR_LIST:
    for quarter in QUARTER.values():
        zip_file_name = f"{year}{quarter}_hokkaido_denkiyohou.zip"
        zip_url = f"{HKD_BASE_URL}/{zip_file_name}"
        response = requests.get(zip_url)

        if response.status_code == 200:
            with open(os.path.join(HKD_PATH, zip_file_name), 'wb') as f:
                f.write(response.content)
            print(f"Downloaded {zip_file_name}")
        else:
            print(f"Failed to download {zip_file_name}")

            time.sleep(np.random.uniform(1, 5))

Downloaded 202004-06_hokkaido_denkiyohou.zip
Downloaded 202007-09_hokkaido_denkiyohou.zip
Downloaded 202010-12_hokkaido_denkiyohou.zip
Failed to download 202001-03_hokkaido_denkiyohou.zip
Downloaded 202104-06_hokkaido_denkiyohou.zip
Downloaded 202107-09_hokkaido_denkiyohou.zip
Downloaded 202110-12_hokkaido_denkiyohou.zip
Downloaded 202101-03_hokkaido_denkiyohou.zip
Downloaded 202204-06_hokkaido_denkiyohou.zip
Downloaded 202207-09_hokkaido_denkiyohou.zip
Downloaded 202210-12_hokkaido_denkiyohou.zip
Downloaded 202201-03_hokkaido_denkiyohou.zip
Downloaded 202304-06_hokkaido_denkiyohou.zip
Downloaded 202307-09_hokkaido_denkiyohou.zip
Downloaded 202310-12_hokkaido_denkiyohou.zip
Downloaded 202301-03_hokkaido_denkiyohou.zip
Downloaded 202404-06_hokkaido_denkiyohou.zip
Downloaded 202407-09_hokkaido_denkiyohou.zip
Downloaded 202410-12_hokkaido_denkiyohou.zip
Downloaded 202401-03_hokkaido_denkiyohou.zip
Downloaded 202504-06_hokkaido_denkiyohou.zip
Downloaded 202507-09_hokkaido_denkiyohou.zip
Do

### Tohoku

In [19]:
THK_PATH = r"/workspace/src/stg/data_lake/electric_forecast/tohoku"
THK_BASE_URL = "https://setsuden.nw.tohoku-epco.co.jp/common/demand/"
THK_YEAR_LIST = [i for i in range(2016,2026)]

In [20]:
for year in THK_YEAR_LIST:
    file_name = f"juyo_{year}_tohoku.csv"
    url = f"{THK_BASE_URL}/{file_name}"
    response = requests.get(url)

    if response.status_code == 200:
        print(f"Processing data for year {year}")
        df = pl.read_csv(source=url, encoding="cp932", skip_rows=1)
        df.write_parquet(os.path.join(THK_PATH, f"juyo_{year}_tohoku.parquet"))
        print(f"Saved parquet for year {year}")
    else:
        print(f"Failed to retrieve data for year {year}")
        continue

    time.sleep(np.random.uniform(1, 5))

Processing data for year 2016
Saved parquet for year 2016
Processing data for year 2017
Saved parquet for year 2017
Processing data for year 2018
Saved parquet for year 2018
Processing data for year 2019
Saved parquet for year 2019
Processing data for year 2020
Saved parquet for year 2020
Processing data for year 2021
Saved parquet for year 2021
Processing data for year 2022
Saved parquet for year 2022
Processing data for year 2023
Saved parquet for year 2023
Processing data for year 2024
Saved parquet for year 2024
Processing data for year 2025
Saved parquet for year 2025


### Chubu

In [21]:
CHB_PATH = r"/workspace/src/stg/data_lake/electric_forecast/chubu/zip"
CHB_BASE_URL = "https://powergrid.chuden.co.jp/denki_yoho_content_data/download_csv/"
CHB_YEAR_LIST = [i for i in range(2019,2026)]
CHB_MONTH_LIST = [i for i in range(1,13)]

In [24]:
#201912_power_usage.zip
for year in CHB_YEAR_LIST:
    for month in CHB_MONTH_LIST:
        zip_file_name = f"{year}{str(month).zfill(2)}_power_usage.zip"
        zip_url = f"{CHB_BASE_URL}/{zip_file_name}"
        time.sleep(np.random.uniform(1, 5))
        response = requests.get(zip_url)
        if response.status_code == 200:
            with open(os.path.join(CHB_PATH, zip_file_name), 'wb') as f:
                f.write(response.content)
            print(f"Downloaded {zip_file_name}")
        else:
            print(f"Failed to download {zip_file_name}")

Failed to download 201901_power_usage.zip
Failed to download 201902_power_usage.zip
Failed to download 201903_power_usage.zip
Downloaded 201904_power_usage.zip
Downloaded 201905_power_usage.zip
Downloaded 201906_power_usage.zip
Downloaded 201907_power_usage.zip
Downloaded 201908_power_usage.zip
Downloaded 201909_power_usage.zip
Downloaded 201910_power_usage.zip
Downloaded 201911_power_usage.zip
Downloaded 201912_power_usage.zip
Downloaded 202001_power_usage.zip
Downloaded 202002_power_usage.zip
Downloaded 202003_power_usage.zip
Downloaded 202004_power_usage.zip
Downloaded 202005_power_usage.zip
Downloaded 202006_power_usage.zip
Downloaded 202007_power_usage.zip
Downloaded 202008_power_usage.zip
Downloaded 202009_power_usage.zip
Downloaded 202010_power_usage.zip
Downloaded 202011_power_usage.zip
Downloaded 202012_power_usage.zip
Downloaded 202101_power_usage.zip
Downloaded 202102_power_usage.zip
Downloaded 202103_power_usage.zip
Downloaded 202104_power_usage.zip
Downloaded 202105_power_